In [1]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math
from sympy import Abs

init_printing(use_latex=True)

In [2]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}',
        latex_str
    )
    # Color all q_i(t) terms orange
    latex_str = re.sub(
        r'(q_\{([1])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{red} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([2])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(q_\{([3])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{yellow} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [3]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

For now, we assume frequencies to be constant in time. Hartwig et. al. did a bit of analysis how this affects the clock noise removal.

In [4]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')
phiREF  = Function(r'\phi_{REF}')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

omegaREFA, omegaREFB, omegaREFC = symbols(r'\omega^{REF}_A \omega^{REF}_B \omega^{REF}_C', real=True)

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')


N1_1 = Function('N_{1_1}') # measured on board 1 coming from laser 1
N2_2 = Function('N_{2_2}')
N3_3 = Function('N_{3_3}')

N1_2 = Function('N_{1_2}')
N1_3 = Function('N_{1_3}')
N2_1 = Function('N_{2_1}')
N2_3 = Function('N_{2_3}')
N3_1 = Function('N_{3_1}')
N3_2 = Function('N_{3_2}')


n1_1 = Function('n_{1_1}') # me asured on board 1 coming from laser 1
n2_2 = Function('n_{2_2}')
n3_3 = Function('n_{3_3}')

n1_2 = Function('n_{1_2}')
n1_3 = Function('n _{1_3}')
n2_1 = Function('n_{2_1}')
n2_3 = Function('n_{2_3}')
n3_1 = Function('n_{3_1}')
n3_2 = Function('n_{3_2}')


N1_m = Function('P_{1}^{m}')
N2_m = Function('P_{2}^{m}')
N3_m = Function('P_{3}^{m}')

In [5]:
def D(expr, tau):
    return expr.subs(t, t - tau)

For the clock jitters, $\epsilon$ variables are used for the delay line boards. The $q$ variables are for the Mokus and in reference to the global time.

In [213]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, omegaREFA, N1_m),
    2: (phi2, q2, omega2, omega2m, epsilonB, omegaREFB, N2_m),
    3: (phi3, q3, omega3, omega3m, epsilonC, omegaREFC, N3_m),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
}



# Numerical ordering frequencies, used only to determine polarity
w_ord  = {1: 17e6, 2: 38e6, 3: 47e6}
wm_ord = {1: 10e6, 2: 11e6, 3: 11e6}

sgn = lambda x: 1 if x > 0 else -1

# Polarity of the carrier and sideband frequency differences
pol_c = {
    (i, j): sgn(w_ord[j] - w_ord[i])
    for (i, j) in tau
}

pol_sb = {
    (i, j): sgn(
        (w_ord[j] + wm_ord[j]) - (w_ord[i]+ wm_ord[i])
    )
    for (i, j) in tau
}


eta    = {}
etaSB  = {}
REF    = {}
r      = {}

include_phi = True
include_clock_noise = False
include_board_jitter = True


for (i, j) in tau:

    phi_i, q_i, om_i, omm_i, eps_i, omegaREFi, N_i_m = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, omegaREFj, N_j_m = sc[j]

    t_ij = tau[(i, j)]

    # FIX 1: the phasemeter flips the WHOLE carrier beat, so pol_c multiplies
    #        both phi_j and phi_i (was only on D(phi_j) before).
    phi_terms = (
        pol_c[(i, j)] * (D(phi_j(t), t_ij) - phi_i(t))
    ) if include_phi else 0

    clock_terms_C = (
        -pol_c[(i, j)] * (om_j - om_i) * q_i(t)
    ) if include_clock_noise else 0

    # FIX 2: pol_sb multiplies the WHOLE sideband clock beat (all three terms,
    #        not just the first).
    clock_terms_SB = (
        pol_sb[(i, j)] * (
            -(om_j - om_i + omm_j - omm_i) * q_i(t)
            - omm_i * q_i(t)
            + omm_j * D(q_j(t), t_ij)
        )
    ) if include_clock_noise else 0

    # DAC emits a POSITIVE frequency, so these are |Δω|, |Δω_SB| by construction.
    delta_om    = pol_c[(i, j)]  * (om_j - om_i)
    delta_om_SB = pol_sb[(i, j)] * (om_j + omm_j - (om_i + omm_i))

    # board jitter from the ADC/DAC chain (kept as YOU derived it from hardware).
    # NOTE: board_terms_SB below mixes the far sideband against the LOCAL SIDEBAND
    #       (om_i + omm_i). If your board actually beats the far SB against the
    #       local CARRIER, change the local ADC term to  -om_i * eps_i(t)  and
    #       delta_om_SB's om_i+omm_i to om_i  -- that is what makes r's board part
    #       come out clean (eps - D eps) instead of carrying an extra omm term.
    board_terms_c = (
        -om_j * D(eps_i(t), t_ij)
        - om_i * eps_i(t)
        + delta_om * eps_i(t)
    ) if include_board_jitter else 0

    board_terms_SB = (
        -(om_j + omm_j) * D(eps_i(t), t_ij)
        - (om_i + omm_i) * eps_i(t)
        + delta_om_SB * eps_i(t)
    ) if include_board_jitter else 0

    eta[(i, j)] = collect(
        expand(phi_terms + clock_terms_C + board_terms_c),
        [q_i(t), q_j(t), eps_i(t)]
    )

    etaSB[(i, j)] = collect(
        expand(phi_terms + clock_terms_SB + board_terms_SB),
        [q_i(t), q_j(t), eps_i(t)]
    )

    r[(i, j)] = simplify(-(eta[(i, j)] - etaSB[(i, j)]) / omm_j)

    REF[(i,j)] = ( - int(include_clock_noise) *q1(t) + int(include_board_jitter) * eps_i(t) ) 

In [214]:
if True:
    for (i, j) in tau:
        display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + color_terms(latex(eta[(i,j)]))))
        display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + color_terms(latex(etaSB[(i,j)]))))
        display(Math(r'\frac{r_{' + str(i) + str(j) + r'}}{\omega_{' + str(j) + '}^m} = ' + color_terms(latex(r[(i,j)]))))
        display(Math(r'\frac{REF_{' + str(i) + str(j) + r'}}{ \omega_{' + str(j) + '}^{10MHz}} = ' + color_terms(latex(REF[(i,j)]))))
        #display(Math(r'\frac{r^{\text{m}}_{' + str(i) + str(j) + r'}}{\omega_{' + str(j) + '}^m} = ' + color_terms(latex(r_m[(i,j)]))))
        print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


In [215]:
print("cleaning the carrieres")

eta[(1,2)] = simplify(eta[(1,2)]   - omega2*(REF[(1,2)]-D(REF[(1,2)], tau[(1,2)])) + 2*omega1*REF[(1,2)]  )
eta[(2,1)] = simplify(-eta[(2,1)]  - omega1*(REF[(2,1)]+D(REF[(2,1)], tau[(2,1)]))  )
eta[(1,3)] = simplify(eta[(1,3)]   - omega3*(REF[(1,3)]-D(REF[(1,3)], tau[(1,3)])) + 2*omega1*REF[(1,3)]  )
eta[(3,1)] = simplify(-eta[(3,1)]  - omega1*(REF[(3,1)]+D(REF[(3,1)], tau[(3,1)]))  )

display(Math(r'\eta_{' + str(1) + str(2) + '} = ' + color_terms(latex(simplify(eta[(1,2)]    )  ))))
display(Math(r'\eta_{21} = ' + color_terms(latex(simplify(eta[(2,1)]  )  ))))


display(Math(r'\eta_{13} = ' + color_terms(latex(simplify(eta[(1,3)]    )  ))))
display(Math(r'\eta_{31} = ' + color_terms(latex(simplify(-eta[(3,1)]     )  ))))


cleaning the carrieres


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

We actually look at first generation TDI here. I do not know how to properly cancel out the timechaning delays with symbolic calculations.

In [209]:
def P12(expr): return expr - D(expr, tau13 + tau31)
def P21(expr): return D(expr, tau12) - D(expr, tau12 + tau13 + tau31)
def P13(expr): return -(expr - D(expr, tau12 + tau21))
def P31(expr): return -(D(expr, tau13) - D(expr, tau13 + tau12 + tau21))

# X1
X1 = collect(expand(P13(eta[(1,3)] + D(eta[(3,1)], tau13))+ P12(eta[(1,2)] + D(eta[(2,1)], tau12))), [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t)])

In [210]:
display(Math(r'X_1 = ' + color_terms(latex(X1)))) 

<IPython.core.display.Math object>

In [211]:
R = {}

R[(1,2)] = -(r[(1,3)] + D(r[(3,1)], tau13))

R[(1,3)] = r[(1,2)] + D(r[(2,1)], tau12)

R[(2,1)] = (r[(1,2)]
             - r[(1,3)]
             - D(r[(3,1)], tau13)
             - D(r[(1,2)], tau13 + tau31))

R[(3,1)] = (-r[(1,3)]
              + r[(1,2)]
              + D(r[(2,1)], tau12)
              + D(r[(1,3)], tau12 + tau21))

R[(2,3)] = 0
R[(3,2)] = 0

In [212]:
a = {
    (1,2): omega1 - omega2,
    (2,1): omega2 - omega1,
    (1,3): omega1 - omega3,
    (3,1): omega3 - omega1,
    (2,3): omega2 - omega3,
    (3,2): omega3 - omega2,
}

# clockwise triplets I+_3
triplets = [(1,2,3), (2,3,1), (3,1,2)]

correction = 0
for (i, j, k) in triplets:
    temp = (- a[(i,j)] * R[(i,j)] - a[(i,k)] * R[(i,k)])
    correction -= temp
    display(Math(r'X_1 = ' + color_terms(latex(collect(temp, [omega1, omega2, omega3])))))


X1c = simplify(X1 - correction) # Added a minus here bc the original r calc was flipped, which i now fixed, but didnt change in the correcting variables

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [198]:
print("The X1 expression is:")
display(Math(r'X_1 = ' + color_terms(latex(collect(X1, [omega1, omega2, omega3])))))
print("The corrected X1 expression is:")
display(Math(r'X_1 = ' + color_terms(latex(X1c))))

The X1 expression is:


<IPython.core.display.Math object>

The corrected X1 expression is:


<IPython.core.display.Math object>

In [199]:
REF_AB= REF[(1,2)] - REF[(2,1)]
REF_AC= REF[(3,1)] - REF[(1,2)]
REF_CB= REF[(2,1)] - REF[(3,1)]

display(Math(r'REF_{AB} = ' + color_terms(latex(REF_AB))))
display(Math(r'REF_{AC} = ' + color_terms(latex(REF_AC))))
display(Math(r'REF_{CB} = ' + color_terms(latex(REF_CB))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [200]:
Corr1 = omega1*(D(REF_AB, tau12) - D(REF_AB, tau12 + tau21) - D(REF_AB, tau12 + tau13 + tau31))
Corr2 = omega1*(D(REF_AC, tau13) - D(REF_AC, tau13 + tau31) - D(REF_AC, tau12 + tau13 + tau21))
Corr3 = -omega1*(D(REF_CB, tau12 + tau13 + tau21 + tau31))

display(Math(r'X_1 = ' + color_terms(latex(simplify(X1c - Corr3)))))

display(Math(r'X_1 = ' + color_terms(latex(simplify(X1c + Corr1 + Corr2)))))
display(Math(r'X_1 = ' + color_terms(latex(simplify(X1c + Corr1 + Corr2 + Corr3)))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [201]:
# When no PM noise
Corr1 = -omega1*(D(REF[(2,1)], tau12) + D(REF[(2,1)], tau12 + tau21) - D(REF[(2,1)], tau12 + tau13 + tau31) - D(REF[(2,1)], tau12 + tau13 + tau21 + tau31))
Corr2 = omega1*(D(REF[(3,1)], tau13) + D(REF[(3,1)], tau13 + tau31) - D(REF[(3,1)], tau12 + tau13 + tau21) - D(REF[(3,1)], tau12 + tau13 + tau21 + tau31))
Corr3 = -omega2*(REF[(1,3)] - D(REF[(1,3)], tau12) - D(REF[(1,3)], tau13 + tau31) + D(REF[(1,3)], tau12 + tau13 + tau31))
Corr4 = -omega3*(-REF[(1,3)] + D(REF[(1,3)], tau13) + D(REF[(1,3)], tau12 + tau21) - D(REF[(1,3)], tau12 + tau13 + tau21))
Corr5 = 2*omega1*D(REF[(1,3)], tau12 + tau21) - 2*omega1*D(REF[(1,3)], tau13 + tau31)



display(Math(r'X_1 = ' + color_terms(latex(REF[(2,1)]))))
display(Math(r'X_1 = ' + color_terms(latex(REF[(3,1)]))))
display(Math(r'X_1 = ' + color_terms(latex(REF[(1,3)]))))
display(Math(r'X_1 = ' + color_terms(latex(simplify(X1 )))))
display(Math(r'X_1 = ' + color_terms(latex(simplify(X1 + Corr1 + Corr2 + Corr3 + Corr4 + Corr5)))))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>